# 01-02 装饰器与生成器

**为什么 Agent 开发需要装饰器和生成器？**

- **装饰器**: 为 LLM 调用添加重试、缓存、日志、限速等能力，不改变原有逻辑
- **生成器**: 处理流式输出（streaming tokens），内存高效地处理大规模数据

**本节目标**：
- 掌握装饰器的原理与常见模式（重试、缓存、计时）
- 理解生成器与 `yield`
- 实战：为 LLM 调用添加自动重试装饰器

---

## Part 1: 装饰器

### 1.1 装饰器原理

In [ ]:
import time
import functools

# 最简单的装饰器：计时
def timer(func):
    @functools.wraps(func)  # 保留原函数的名称和文档
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        elapsed = time.time() - start
        print(f"[Timer] {func.__name__} 耗时 {elapsed:.3f}s")
        return result
    return wrapper

@timer
def slow_function(n: int) -> int:
    time.sleep(0.1)
    return n * 2

result = slow_function(5)
print(f"结果: {result}")

In [ ]:
# 带参数的装饰器
def retry(max_attempts: int = 3, delay: float = 1.0, exceptions=(Exception,)):
    """自动重试装饰器 —— Agent 调用外部工具时必备"""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_error = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    last_error = e
                    print(f"[Retry] 第 {attempt}/{max_attempts} 次失败: {e}")
                    if attempt < max_attempts:
                        time.sleep(delay)
            raise last_error
        return wrapper
    return decorator

import random

@retry(max_attempts=3, delay=0.1)
def unstable_api_call(query: str) -> str:
    """模拟不稳定的外部 API"""
    if random.random() < 0.6:  # 60% 失败率
        raise ConnectionError("API 连接超时")
    return f"API 返回: {query} 的分析结果"

try:
    result = unstable_api_call("广告点击率分析")
    print(f"成功: {result}")
except Exception as e:
    print(f"最终失败: {e}")

In [ ]:
# 缓存装饰器 —— 避免重复调用昂贵的 LLM
from functools import lru_cache

@lru_cache(maxsize=128)
def cached_llm_response(prompt: str) -> str:
    """相同 prompt 只调用一次，后续从缓存返回"""
    time.sleep(0.5)  # 模拟 LLM 调用延迟
    return f"[LLM 响应] {prompt[:20]}..."

# 第一次调用
start = time.time()
r1 = cached_llm_response("什么是 AI Agent？请详细解释")
print(f"第1次: {time.time()-start:.3f}s — {r1}")

# 第二次调用（命中缓存，几乎瞬间）
start = time.time()
r2 = cached_llm_response("什么是 AI Agent？请详细解释")
print(f"第2次: {time.time()-start:.3f}s — {r2} (来自缓存)")
print(f"缓存命中: {cached_llm_response.cache_info()}")

### 1.2 类装饰器 —— 管理状态

In [ ]:
class RateLimiter:
    """限速装饰器 —— 控制 LLM API 调用频率（避免超出 RPM 限制）"""
    
    def __init__(self, calls_per_second: float = 1.0):
        self.min_interval = 1.0 / calls_per_second
        self.last_call_time = 0.0
    
    def __call__(self, func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            elapsed = time.time() - self.last_call_time
            wait_time = self.min_interval - elapsed
            if wait_time > 0:
                print(f"[RateLimit] 等待 {wait_time:.2f}s...")
                time.sleep(wait_time)
            self.last_call_time = time.time()
            return func(*args, **kwargs)
        return wrapper

rate_limiter = RateLimiter(calls_per_second=2)  # 最多每秒 2 次

@rate_limiter
def api_call(i: int) -> str:
    return f"调用 {i} 完成"

for i in range(4):
    print(api_call(i))

## Part 2: 生成器

### 2.1 基础生成器

In [ ]:
# 普通函数 vs 生成器函数
def list_numbers(n: int) -> list:
    return [i * i for i in range(n)]  # 一次性生成，全部存入内存

def gen_numbers(n: int):
    for i in range(n):
        yield i * i  # 按需生成，一次一个

# 对比内存
import sys
lst = list_numbers(10000)
gen = gen_numbers(10000)
print(f"list 占用: {sys.getsizeof(lst):,} bytes")
print(f"generator 占用: {sys.getsizeof(gen):,} bytes")

# 使用生成器
for i, val in enumerate(gen_numbers(5)):
    print(f"  第 {i} 个: {val}")

In [ ]:
# 模拟 LLM 流式输出（streaming tokens）
def stream_llm_response(prompt: str):
    """生成器模拟 LLM 的流式 token 输出"""
    # 真实场景中这里是 openai.chat.completions.create(stream=True)
    words = f"这是对'{prompt}'的回答：AI Agent 是能够感知环境、做出决策、执行动作的智能体。".split()
    for word in words:
        time.sleep(0.05)  # 模拟 token 生成延迟
        yield word + " "

print("流式输出: ", end="", flush=True)
for token in stream_llm_response("什么是 AI Agent"):
    print(token, end="", flush=True)
print()  # 换行

In [ ]:
# 数据管道：多个生成器串联
def read_documents(paths: list[str]):
    """生成文档内容"""
    for path in paths:
        yield {"path": path, "content": f"这是 {path} 的内容..."}

def chunk_documents(documents, chunk_size: int = 100):
    """切分文档（RAG 预处理步骤）"""
    for doc in documents:
        content = doc["content"]
        for i in range(0, len(content), chunk_size):
            yield {"path": doc["path"], "chunk": content[i:i+chunk_size], "chunk_id": i//chunk_size}

def add_metadata(chunks):
    """添加元数据"""
    for chunk in chunks:
        chunk["length"] = len(chunk["chunk"])
        yield chunk

# 管道执行：内存中同时只有一个 chunk
fake_docs = ["doc1.pdf", "doc2.pdf", "doc3.pdf"]
pipeline = add_metadata(chunk_documents(read_documents(fake_docs)))

for i, chunk in enumerate(pipeline):
    print(f"Chunk {i}: {chunk}")
    if i > 3: break

## 总结

### 装饰器常用场景
| 装饰器 | 场景 |
|--------|------|
| `@retry` | LLM 调用/工具调用失败重试 |
| `@lru_cache` | 缓存相同 prompt 的响应 |
| `@timer` | 性能分析 |
| `@RateLimiter` | 控制 API 调用频率 |

### 生成器常用场景
| 生成器 | 场景 |
|--------|------|
| streaming | LLM 流式输出处理 |
| 数据管道 | 大规模文档预处理（RAG） |
| 懒加载 | 按需读取大文件 |

**自检**: 能写一个带重试和日志的装饰器，并用生成器实现一个简单的文档分块管道吗？

**下一节**: `03_type_hints_pydantic.ipynb` — 类型注解与 Pydantic